In [1]:
# Install required packages
!pip install streamlit PyMuPDF google-genai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.4 MB/s eta 0:00:00


In [3]:
# Set your API key (you can also use Colab secrets)
from google.colab import userdata
import os
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

In [13]:
# Create the app.py file
%%writefile app.py
# [Paste the complete code here]
import re
import fitz
import json
import streamlit as st
from google import genai
from google.genai import types
from io import BytesIO
import traceback
from datetime import datetime
import os
import sys

# CONFIGURATION
# Load API key - for Colab, you can set this as a secret or environment variable
API_KEY = os.getenv('GEMINI_API_KEY')  # Replace with your actual API key
MODEL_CTX = "gemini-2.0-flash"
MODEL_MAP = "gemini-2.0-flash"

REFERRAL_PROMPT = """
You are a medical information extraction specialist. You have two PDFs:
- PDF1: A complete medical referral package containing patient demographics, medical history, diagnoses, medications, lab results, and clinical notes
- PDF2: A Prior Authorization (PA) form that needs to be filled out

Your task is to carefully examine EVERY PAGE of the referral package and extract ALL relevant patient information that could be used to fill out the PA form.

EXTRACTION REQUIREMENTS:
1. Patient Demographics: Full name, DOB, address, phone numbers, insurance information
2. Medical Information: All diagnoses (with ICD codes if available), current medications, allergies, lab results
3. Clinical History: Disease severity, previous treatments, treatment failures, adverse reactions
4. Provider Information: Prescribing physician details, administration location
5. Treatment Details: Requested medication, dosage, administration method

Pay special attention to:
- Checkbox-related information (Yes/No answers about previous treatments, contraindications, etc.)
- Medical necessity criteria
- Previous medication trials and their outcomes
- Disease activity and severity indicators
- Laboratory values and test results

Structure your response as a comprehensive JSON object with clear categories:
{
  "patient_demographics": {
    "first_name": "...",
    "last_name": "...",
    "date_of_birth": "...",
    "address": "...",
    "phone": "...",
    "member_id": "...",
    "insurance_info": "..."
  },
  "medical_information": {
    "primary_diagnosis": "...",
    "icd_codes": ["..."],
    "secondary_diagnoses": ["..."],
    "current_medications": ["..."],
    "allergies": ["..."],
    "weight": "...",
    "height": "..."
  },
  "clinical_history": {
    "disease_severity": "...",
    "previous_treatments": [
      {
        "medication": "...",
        "outcome": "...",
        "reason_for_discontinuation": "...",
        "dates": "..."
      }
    ],
    "contraindications": ["..."],
    "lab_results": "...",
    "disease_activity": "..."
  },
  "provider_information": {
    "prescriber_name": "...",
    "prescriber_credentials": "...",
    "npi": "...",
    "address": "...",
    "phone": "...",
    "administration_location": "..."
  },
  "treatment_request": {
    "requested_medication": "...",
    "dose": "...",
    "frequency": "...",
    "route": "...",
    "indication": "..."
  }
}

Extract information even if it's not perfectly formatted - look for partial matches and clinical context clues.
"""

# Initialize Gemini client
try:
    client = genai.Client(api_key=API_KEY)
    st.sidebar.success("Gemini API configured successfully")
except Exception as e:
    client = None
    st.sidebar.error(f"Gemini API configuration failed: {e}")

# UTILITIES
def extract_json(text):
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("No JSON Object found")
    raw = text[start:end+1]
    raw = re.sub(r'(?m)^\s*([A-Za-z0-9_]+)\s*:', r'"\1":', raw)
    raw = re.sub(r',\s*([}\]])', r'\1', raw)
    return raw

def extract_patient_info(referral_bytes, pa_bytes):
    """Runs the user provided Gemini prompt to extract patient info"""
    if not client:
        raise Exception("Gemini client not initialized. Please check your API key.")

    pdf1 = types.Part.from_bytes(data=referral_bytes, mime_type='application/pdf')
    pdf2 = types.Part.from_bytes(data=pa_bytes, mime_type='application/pdf')
    response = client.models.generate_content(
        model=MODEL_CTX,
        contents=[pdf1, pdf2, REFERRAL_PROMPT]
    ).text
    print(response)
    raw = extract_json(response)
    return json.loads(raw)

def make_page_part(pdf_bytes, page_no):
    """
    Create a one-page PDF part from the given PDF bytes.
    Tries to copy the form page; on XRef errors, falls back to image-based PDF.
    """
    src = fitz.open(stream=pdf_bytes, filetype='pdf')
    try:
        dst = fitz.open()
        dst.insert_pdf(src, from_page=page_no-1, to_page=page_no-1)
        for w in dst[0].widgets() or []:
            dst[0].delete_widget(w)
        buf = BytesIO()
        dst.save(buf)
        src.close()
        dst.close()
        return types.Part.from_bytes(data=buf.getvalue(), mime_type='application/pdf')
    except Exception:
        page = src[page_no-1]
        pix = page.get_pixmap()
        new_pdf = fitz.open()
        rect = page.rect
        new_page = new_pdf.new_page(width=rect.width, height=rect.height)
        new_page.insert_image(rect, pixmap=pix)
        buf = BytesIO()
        new_pdf.save(buf)
        src.close()
        new_pdf.close()
        return types.Part.from_bytes(data=buf.getvalue(), mime_type='application/pdf')

def extract_fields_with_positions(pdf_bytes):
    doc = fitz.open(stream=pdf_bytes, filetype='pdf')
    fields = []
    for page_num, page in enumerate(doc, start=1):
        for w in page.widgets() or []:
            fields.append({
                "name": w.field_name,
                "type": "checkbox" if w.field_type == fitz.PDF_WIDGET_TYPE_CHECKBOX else "text",
                "value": w.field_value,
                "page": page_num,
                "rect": list(map(float, w.rect)),
            })
    doc.close()
    return fields

def get_field_context_for_page(pa_bytes, page_no, page_fields):
    """Extract context for fields on a specific page"""
    if not client:
        return []

    page_part = make_page_part(pa_bytes, page_no)

    prompt_ctx = f"""
    You are analyzing page {page_no} of a medical Prior Authorization form for rituximab/biosimilar medications.

    Your task: For each form field, identify the EXACT question being asked and provide relevant medical context.

    FIELD ANALYSIS REQUIREMENTS:
    1. Read the form carefully to understand each field's purpose
    2. For checkbox fields, identify what "Yes" or "No" would mean
    3. For text fields, identify what type of information is expected
    4. Consider the medical context (this is for rituximab PA requests)

    MEDICAL CONTEXT AWARENESS:
    - This form is for rituximab and biosimilar medications
    - Common conditions: B-cell lymphomas, rheumatoid arthritis, autoimmune conditions
    - Prior authorization requires evidence of medical necessity
    - Fields often ask about previous treatment failures or contraindications

    For each field in the provided list, return a JSON object with:
    - "name": field identifier (keep exact as provided)
    - "page": page number
    - "question": The EXACT question or label associated with this field on the form
    - "context": Medical context explaining what this field is asking for (25 words max)
    - "field_type": "checkbox", "text", "radio", or "dropdown"
    - "expected_values": For checkboxes: ["Yes", "No"], for text: ["free_text"] or specific options if visible

    EXAMPLE OUTPUT FORMAT:
    [
      {{
        "name": "CB1",
        "page": 2,
        "question": "Has the patient had prior therapy with the requested product within the last 365 days?",
        "context": "Determines if this is continuation therapy or new treatment initiation",
        "field_type": "checkbox",
        "expected_values": ["Yes", "No"]
      }},
      {{
        "name": "T1",
        "page": 2,
        "question": "First Name:",
        "context": "Patient's legal first name as it appears on insurance card",
        "field_type": "text",
        "expected_values": ["free_text"]
      }}
    ]

    Here are the fields to analyze:
    {json.dumps(page_fields, indent=2)}

    Return ONLY valid JSON array. Be precise with question text as it appears on the form.
    """

    try:
        response = client.models.generate_content(
            model=MODEL_CTX,
            contents=[page_part, prompt_ctx]
        ).text
        print(f"Context response for page {page_no}: {response}")

        # Try to extract JSON array
        start = response.find("[")
        end = response.rfind("]")
        if start != -1 and end != -1:
            raw = response[start:end+1]
            return json.loads(raw)
        else:
            # Fallback to object extraction
            raw = extract_json(response)
            data = json.loads(raw)
            return [data] if isinstance(data, dict) else data

    except Exception as e:
        print(f"Error getting context for page {page_no}: {e}")
        return []

def map_patient_info_to_fields(patient_info, field_contexts):
    """Map extracted patient information to form fields using AI"""
    if not client:
        return {}

    mapping_prompt = f"""
    You are a medical form filling specialist with expertise in Prior Authorization forms for rituximab medications.

    TASK: Map extracted patient information to specific form fields with high accuracy.

    PATIENT INFORMATION:
    {json.dumps(patient_info, indent=2)}

    FORM FIELDS TO FILL:
    {json.dumps(field_contexts, indent=2)}

    MAPPING RULES AND EXPERTISE:

    1. DEMOGRAPHIC FIELDS:
       - Match names, DOB, addresses, phone numbers exactly
       - Insurance ID numbers must match exactly
       - Use proper date formats (MM/DD/YYYY for most US forms)

    2. CHECKBOX FIELDS - CRITICAL LOGIC:
       - "Yes" = condition is present, treatment was tried, adverse reaction occurred
       - "No" = condition is absent, treatment was not tried, no adverse reaction
       - Only check "Yes" if you have clear evidence from patient data
       - If uncertain, prefer "No" for safety (can be corrected manually)

    3. MEDICAL CONDITION FIELDS:
       - Match ICD codes when available
       - Map diagnoses to specific condition categories
       - Consider synonyms (e.g., "RA" = "rheumatoid arthritis")

    4. PREVIOUS TREATMENT ANALYSIS:
       - Look for evidence of prior medication trials
       - Identify reasons for discontinuation (ineffective, adverse reaction, contraindication)
       - Map specific medication names to form options

    5. CLINICAL SEVERITY:
       - Map clinical descriptions to severity levels (mild/moderate/severe)
       - Look for disease activity indicators
       - Consider lab values and clinical assessments

    SPECIFIC RITUXIMAB PA LOGIC:
    - If patient has B-cell lymphoma → check appropriate lymphoma type
    - If patient has RA → verify severity and previous DMARD failures
    - If requesting biosimilar → check for contraindications to preferred products
    - Previous rituximab use → affects "prior therapy" questions

    CONFIDENCE REQUIREMENTS:
    - Only fill fields where you have 80%+ confidence
    - For ambiguous cases, provide the most likely answer but note uncertainty
    - Use clinical judgment based on medical context

    OUTPUT FORMAT:
    Return ONLY a JSON object mapping field names to values:
    {{
      "field_name": "value",
      "T1": "John",
      "T2": "Doe",
      "CB_prior_therapy": "Yes",
      "dropdown_diagnosis": "Diffuse large B-cell lymphoma"
    }}

    VALUE FORMATTING:
    - Text fields: Use exact text (names, addresses, etc.)
    - Checkboxes: "Yes" or "No" only
    - Dates: MM/DD/YYYY format
    - Numbers: Use appropriate units (lbs, kg, etc.)

    MEDICAL ACCURACY IS CRITICAL - Only map fields you're confident about based on the patient data.
    """

    try:
        response = client.models.generate_content(
            model=MODEL_MAP,
            contents=[mapping_prompt]
        ).text
        print(f"Mapping response: {response}")
        raw = extract_json(response)
        return json.loads(raw)
    except Exception as e:
        print(f"Error in mapping: {e}")
        return {}

def fill_pdf_form(pdf_bytes, field_mappings):
    """Fill the PDF form with mapped values"""
    doc = fitz.open(stream=pdf_bytes, filetype='pdf')

    filled_count = 0
    total_fields = 0

    for page_num, page in enumerate(doc):
        for widget in page.widgets() or []:
            total_fields += 1
            field_name = widget.field_name

            if field_name in field_mappings and field_mappings[field_name]:
                try:
                    value = field_mappings[field_name]

                    if widget.field_type == fitz.PDF_WIDGET_TYPE_CHECKBOX:
                        if str(value).lower() in ['yes', 'true', '1', 'checked']:
                            widget.field_value = True
                            filled_count += 1
                        elif str(value).lower() in ['no', 'false', '0', 'unchecked']:
                            widget.field_value = False
                            filled_count += 1
                    else:
                        widget.field_value = str(value)
                        filled_count += 1

                    widget.update()

                except Exception as e:
                    print(f"Error filling field {field_name}: {e}")

    output_buffer = BytesIO()
    doc.save(output_buffer)
    doc.close()

    print(f"Filled {filled_count} out of {total_fields} fields")
    return output_buffer.getvalue(), filled_count, total_fields

def validate_and_review_mappings(field_mappings, field_contexts, patient_info):
    """Validate mappings and provide review summary"""
    validation_results = {
        'high_confidence': [],
        'medium_confidence': [],
        'low_confidence': [],
        'unmapped_fields': []
    }

    for field_context in field_contexts:
        field_name = field_context.get('name', '')
        field_question = field_context.get('question', '') or ''  # Ensure it's not None
        field_context_text = field_context.get('context', '') or ''  # Ensure it's not None

        if field_name in field_mappings and field_mappings[field_name]:
            confidence = 'medium'
            value = field_mappings[field_name]

            # Safe string operations with None checks
            field_question_lower = field_question.lower() if field_question else ''

            if any(keyword in field_question_lower for keyword in ['name', 'date', 'id', 'phone']):
                confidence = 'high'
            elif any(keyword in field_question_lower for keyword in ['diagnosis', 'medication', 'dosage', 'clinical']):
                if len(str(value)) < 3:
                    confidence = 'low'

            validation_results[f'{confidence}_confidence'].append({
                'field': field_name,
                'question': field_question,
                'value': value,
                'context': field_context_text
            })
        else:
            validation_results['unmapped_fields'].append({
                'field': field_name,
                'question': field_question,
                'context': field_context_text
            })

    return validation_results

# Helper functions to add:

def reset_session_state():
    """Reset all session state variables"""
    keys_to_reset = [
        'processing_step', 'patient_info', 'field_contexts',
        'field_mappings', 'validation_results', 'pa_bytes',
        'ref_bytes', 'fields'
    ]
    for key in keys_to_reset:
        if key in st.session_state:
            del st.session_state[key]
    st.session_state.processing_step = 0

def get_step_name(step):
    """Get human-readable step name"""
    step_names = {
        1: "Extracting patient information",
        2: "Analyzing form fields",
        3: "Mapping patient data to fields",
        4: "Validating mappings",
        5: "Filling PDF form"
    }
    return step_names.get(step, "Unknown step")

def process_step_1():
    """Step 1: Extract patient information"""
    if st.session_state.patient_info is None:
        with st.spinner("🔍 Extracting patient information from referral package..."):
            try:
                st.session_state.patient_info = extract_patient_info(
                    st.session_state.ref_bytes,
                    st.session_state.pa_bytes
                )
                st.success("✅ Patient information extracted successfully")
            except Exception as e:
                st.error(f"❌ Error extracting patient info: {e}")
                st.session_state.processing_step = 0
                return

    with st.expander("📊 Extracted Patient Information", expanded=True):
        st.json(st.session_state.patient_info)

    st.session_state.processing_step = 2
    st.rerun()

def process_step_2():
    """Step 2: Extract and analyze form fields"""
    if st.session_state.fields is None:
        with st.spinner("📝 Analyzing PA form fields..."):
            try:
                st.session_state.fields = extract_fields_with_positions(st.session_state.pa_bytes)
                st.success(f"✅ Found {len(st.session_state.fields)} form fields")
            except Exception as e:
                st.error(f"❌ Error analyzing form fields: {e}")
                st.session_state.processing_step = 0
                return

    if st.session_state.field_contexts is None:
        with st.spinner("🧠 Extracting field contexts..."):
            try:
                # Group fields by page
                fields_by_page = {}
                for f in st.session_state.fields:
                    fields_by_page.setdefault(f["page"], []).append({
                        "name": f["name"],
                        "type": f["type"],
                        "rect": f["rect"],
                    })

                # Get field contexts page by page
                all_field_contexts = []
                total_pages = len(fields_by_page)

                for i, (page_no, page_fields) in enumerate(sorted(fields_by_page.items())):
                    st.info(f"Processing page {page_no} ({i+1}/{total_pages})")
                    page_contexts = get_field_context_for_page(
                        st.session_state.pa_bytes, page_no, page_fields
                    )
                    all_field_contexts.extend(page_contexts)

                st.session_state.field_contexts = all_field_contexts
                st.success(f"✅ Extracted contexts for {len(all_field_contexts)} fields")

            except Exception as e:
                st.error(f"❌ Error extracting field contexts: {e}")
                st.session_state.processing_step = 0
                return

    st.session_state.processing_step = 3
    st.rerun()

def process_step_3():
    """Step 3: Map patient information to form fields"""
    if st.session_state.field_mappings is None:
        with st.spinner("🔗 Mapping patient information to form fields..."):
            try:
                st.session_state.field_mappings = map_patient_info_to_fields_optimized(
                    st.session_state.patient_info,
                    st.session_state.field_contexts
                )
                st.success("✅ Field mapping completed")
            except Exception as e:
                st.error(f"❌ Error mapping fields: {e}")
                st.session_state.processing_step = 0
                return

    st.session_state.processing_step = 4
    st.rerun()

def process_step_4(review_mode):
    """Step 4: Validate mappings"""
    if st.session_state.validation_results is None:
        with st.spinner("🔍 Validating field mappings..."):
            try:
                st.session_state.validation_results = validate_and_review_mappings(
                    st.session_state.field_mappings,
                    st.session_state.field_contexts,
                    st.session_state.patient_info
                )
                st.success("✅ Validation completed")
            except Exception as e:
                st.error(f"❌ Error validating mappings: {e}")
                st.session_state.processing_step = 0
                return

def process_step_5():
    """Step 5: Fill PDF form and show results"""
    with st.spinner("📝 Filling PDF form..."):
        try:
            filled_pdf_bytes, filled_count, total_fields = fill_pdf_form(
                st.session_state.pa_bytes,
                st.session_state.field_mappings
            )

            st.success(f"🎉 Successfully filled {filled_count} out of {total_fields} form fields!")

            # Summary statistics
            col1, col2, col3 = st.columns(3)
            with col1:
                st.metric("📝 Fields Filled", filled_count)
            with col2:
                st.metric("📊 Total Fields", total_fields)
            with col3:
                fill_rate = (filled_count / total_fields * 100) if total_fields > 0 else 0
                st.metric("📈 Fill Rate", f"{fill_rate:.1f}%")

            # Download button for filled form
            st.download_button(
                label="📥 Download Filled PA Form",
                data=filled_pdf_bytes,
                file_name=f"filled_pa_form_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf",
                mime="application/pdf",
                type="primary"
            )

            # Reset to allow new processing
            if st.button("🔄 Process Another Form"):
                reset_session_state()
                st.rerun()

        except Exception as e:
            st.error(f"❌ Error filling PDF form: {e}")
            with st.expander("🔍 Full error details"):
                st.code(traceback.format_exc())
            st.session_state.processing_step = 0

def show_review_interface():
    """Show the review interface for field mappings"""
    st.subheader("🔍 Field Mapping Review")

    validation_results = st.session_state.validation_results

    col1, col2, col3 = st.columns(3)

    with col1:
        if validation_results['high_confidence']:
            st.success(f"**✅ High Confidence ({len(validation_results['high_confidence'])})**")
            for item in validation_results['high_confidence']:
                st.write(f"• {item['field']}: {item['question']} → **{item['value']}**")

    with col2:
        if validation_results['medium_confidence']:
            st.warning(f"**⚠️ Medium Confidence ({len(validation_results['medium_confidence'])})**")
            for item in validation_results['medium_confidence']:
                st.write(f"• {item['field']}: {item['question']} → *{item['value']}*")

    with col3:
        if validation_results['low_confidence']:
            st.error(f"**❌ Low Confidence ({len(validation_results['low_confidence'])})**")
            for item in validation_results['low_confidence']:
                st.write(f"• {item['field']}: {item['question']} → ⚠️ *{item['value']}*")

    if validation_results['unmapped_fields']:
        with st.expander(f"❓ Unmapped Fields ({len(validation_results['unmapped_fields'])})"):
            for item in validation_results['unmapped_fields']:
                st.write(f"• {item['field']}: {item['question']}")

def map_patient_info_to_fields_optimized(patient_info, field_contexts):
    """Optimized mapping with batching to reduce processing time"""
    if not client:
        return {}

    # Process in smaller batches to avoid timeout
    batch_size = 30  # Reduced batch size for better performance
    all_mappings = {}

    total_batches = (len(field_contexts) + batch_size - 1) // batch_size

    for i in range(0, len(field_contexts), batch_size):
        batch = field_contexts[i:i+batch_size]
        batch_num = i // batch_size + 1

        st.info(f"Processing batch {batch_num}/{total_batches} ({len(batch)} fields)")

        mapping_prompt = f"""
        You are mapping extracted patient information to Prior Authorization form fields.

        Patient Information:
        {json.dumps(patient_info, indent=2)}

        Form Fields with Context (Batch {batch_num}):
        {json.dumps(batch, indent=2)}

        Instructions:
        1. For each form field, determine if any patient information matches
        2. Return ONLY confident mappings (70%+ confidence)
        3. Use "Yes"/"No" for checkboxes, exact text values for text fields
        4. Return empty object {{}} if no confident mappings found

        Return JSON object mapping field names to values:
        {{"field_name": "field_value"}}
        """

        try:
            response = client.models.generate_content(
                model=MODEL_MAP,
                contents=[mapping_prompt]
            ).text

            raw = extract_json(response)
            batch_mappings = json.loads(raw)
            all_mappings.update(batch_mappings)

        except Exception as e:
            st.warning(f"Error in batch {batch_num}: {e}")
            continue

    return all_mappings

# Streamlit configuration for Colab
st.set_page_config(
    page_title="PAFill - Medical Form Automation",
    page_icon="🏥",
    layout="wide",
    initial_sidebar_state="expanded"
)

def main():
    st.title("🏥 PAFill - Automate Insurance Forms")

    # Initialize session state
    if 'processing_step' not in st.session_state:
        st.session_state.processing_step = 0
    if 'patient_info' not in st.session_state:
        st.session_state.patient_info = None
    if 'field_contexts' not in st.session_state:
        st.session_state.field_contexts = None
    if 'field_mappings' not in st.session_state:
        st.session_state.field_mappings = None
    if 'validation_results' not in st.session_state:
        st.session_state.validation_results = None
    if 'pa_bytes' not in st.session_state:
        st.session_state.pa_bytes = None
    if 'ref_bytes' not in st.session_state:
        st.session_state.ref_bytes = None
    if 'fields' not in st.session_state:
        st.session_state.fields = None

    st.write("Upload medical referral packages and Prior Authorization forms to automatically extract and fill patient information.")

    # API Key configuration in sidebar
    st.sidebar.header("⚙️ Configuration")

    # Allow API key input in Colab
    api_key_input = st.sidebar.text_input(
        "Gemini API Key",
        value=API_KEY if API_KEY != "your_api_key_here" else "",
        type="password",
        help="Enter your Google Gemini API key"
    )

    if api_key_input and api_key_input != API_KEY:
        global client
        try:
            client = genai.Client(api_key=api_key_input)
            st.sidebar.success("✅ API Key updated successfully")
        except Exception as e:
            st.sidebar.error(f"❌ API Key error: {e}")
            client = None

    # Processing options
    st.sidebar.header("🔧 Processing Options")
    review_mode = st.sidebar.checkbox("Review mappings before filling", value=True)
    confidence_threshold = st.sidebar.slider("Confidence threshold", 0.0, 1.0, 0.7)

    # Add reset button
    if st.sidebar.button("🔄 Reset Process"):
        reset_session_state()
        st.rerun()

    # File uploaders
    col1, col2 = st.columns(2)

    with col1:
        st.subheader("📄 PA Form")
        pa_file = st.file_uploader("Upload PA form PDF", type=["pdf"], key="pa_form")

    with col2:
        st.subheader("📋 Referral Package")
        ref_file = st.file_uploader("Upload Referral Package PDF", type=["pdf"], key="ref_package")

    # Show current processing status
    if st.session_state.processing_step > 0:
        st.info(f"🔄 Processing Step: {get_step_name(st.session_state.processing_step)}")

    # Main processing button
    if st.button("🚀 Process and Fill Forms", type="primary") and st.session_state.processing_step == 0:
        if not client:
            st.error("❌ Please configure your Gemini API key first")
            st.stop()

        if not pa_file or not ref_file:
            st.error("❌ Please upload both PA form and referral package PDFs")
            st.stop()

        # Store file bytes in session state
        st.session_state.pa_bytes = pa_file.read()
        st.session_state.ref_bytes = ref_file.read()
        st.session_state.processing_step = 1
        st.rerun()

    # Process based on current step
    if st.session_state.processing_step == 1:
        process_step_1()
    elif st.session_state.processing_step == 2:
        process_step_2()
    elif st.session_state.processing_step == 3:
        process_step_3()
    elif st.session_state.processing_step == 4:
        process_step_4(review_mode)
    elif st.session_state.processing_step == 5:
        process_step_5()

    # Show review interface if we're at step 4 and review mode is enabled
    if st.session_state.processing_step == 4 and review_mode and st.session_state.validation_results:
        show_review_interface()

        if st.button("✅ Proceed with Form Filling", type="primary"):
            st.session_state.processing_step = 5
            st.rerun()
    elif st.session_state.processing_step == 4 and not review_mode:
        # Auto-proceed if review mode is disabled
        st.session_state.processing_step = 5
        st.rerun()

if __name__ == "__main__":
    main()

Overwriting app.py


In [14]:
!npm install localtunnel

⠙⠹⠸⠼⠴
up to date, audited 23 packages in 1s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠴

In [15]:
!streamlit run app.py &>/content/logs.txt & npx localtunnel --port 8501 & curl ipv4.icanhazip.com

34.105.123.227
⠙your url is: https://purple-ghosts-punch.loca.lt
